# 12a — Validation Evaluation

This notebook performs Week 6 validation by rebuilding the **Week 5 tuned pipeline as exactly as possible from the saved Week 5 artifacts and shared modeling configuration**.

### Week 5 consistency requirements enforced
- Hyperparameters come from `candidate_models.csv` first, with `hyperparameter_tuning_summary.csv` only as a fallback.
- XGBoost candidates are recreated with `XGBClassifier`.
- The Week 5 pipeline order is restored exactly as:
  `preprocessing → variance_filter → feature_selection → classifier`.
- `VarianceThreshold(threshold=0.0)` is restored before `SelectKBest`.
- `feature_selection__k` is required and restored from the saved Week 5 parameter dictionary.
- Fixed classifier settings come from the shared `src.modeling.config` values used by the project modeling framework.
- Shared preprocessing comes from `src.modeling.preprocessing.prepare_dataset`.
- All preprocessing, variance filtering, feature selection, and classifier fitting occur on the training data only.
- Held-out validation data are used only for prediction/evaluation.
- Both predicted classes and predicted probabilities are saved for Week 6 calibration and error-analysis tasks.
- The notebook performs explicit audits and fails instead of silently continuing when a Week 5 setting cannot be recovered.

### Required outputs
- `outputs/metrics/validation_results.csv`
- `outputs/metrics/validation_predictions.csv`
- `outputs/metrics/validation_probabilities.csv`
- validation confusion matrices
- `outputs/tables/validation_comparison.csv`
- `outputs/tables/validation_best_models.csv`
- `outputs/tables/week5_pipeline_exactness_audit.csv`
- `outputs/figures/validation_model_comparison.png`

> The purpose of this notebook is validation, not retuning. No Week 5 hyperparameter is re-estimated from the validation set.


## 1. Setup

In [ ]:
from pathlib import Path
import ast
import json
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is required because Week 5 includes tuned XGBoost models. "
        "Install it with `pip install xgboost`."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the project repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reuse the shared modeling framework instead of rebuilding preprocessing here.
from src.modeling.preprocessing import prepare_dataset
from src.modeling.evaluation import calculate_metrics
from src.modeling.config import (
    RANDOM_STATE,
    USE_CLASS_WEIGHT,
    CLASS_WEIGHT,
    LOGISTIC_MAX_ITER,
)

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for folder in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Using shared preprocessing from src/modeling/preprocessing.py")
print("Using shared metrics from src/modeling/evaluation.py")


## 2. Load the Week 5 tuning outputs

In [ ]:
week5_files = {
    "candidate_models": METRICS_DIR / "candidate_models.csv",
    "tuned_models": METRICS_DIR / "tuned_models.csv",
    "hyperparameter_tuning_summary": METRICS_DIR / "hyperparameter_tuning_summary.csv",
}

week5_tables = {}

for name, path in week5_files.items():
    if path.exists():
        week5_tables[name] = pd.read_csv(path)
        print(f"✓ Loaded {path.name}: {len(week5_tables[name])} row(s)")
    else:
        print(f"— Not found: {path.name}")

if not week5_tables:
    raise FileNotFoundError(
        "No Week 5 tuning CSV files were found in outputs/metrics."
    )

for name, table in week5_tables.items():
    print(f"\n{name}.csv")
    display(table)

## 3. Build the tuned-model table

Hyperparameters are loaded from the Week 5 output that actually contains the saved **Best Parameters** column.

Priority:
1. `candidate_models.csv`
2. `hyperparameter_tuning_summary.csv`

`tuned_models.csv` is used only as optional model-file metadata because it may contain model names and filenames without the tuned parameter dictionaries.


In [ ]:
def normalize_col_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower()
    ).strip("_")


def find_col(df, choices):
    normalized = {
        normalize_col_name(c): c
        for c in df.columns
    }

    for choice in choices:
        key = normalize_col_name(choice)
        if key in normalized:
            return normalized[key]

    return None


def parse_params(value):
    if value is None or (
        isinstance(value, float)
        and pd.isna(value)
    ):
        return {}

    if isinstance(value, dict):
        return value

    text = str(value).strip()

    if not text:
        return {}

    for parser in (ast.literal_eval, json.loads):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    raise ValueError(
        f"Could not parse saved parameter dictionary: {text[:120]}"
    )


def standardize_model_table(df, require_params=False):
    model_col = find_col(
        df,
        [
            "model",
            "model_name",
            "classifier",
            "estimator",
            "candidate_model",
        ],
    )

    dataset_col = find_col(
        df,
        [
            "dataset",
            "dataset_name",
            "modality",
            "feature_set",
        ],
    )

    params_col = find_col(
        df,
        [
            "Best Parameters",
            "best_params",
            "best_parameters",
            "parameters",
            "params",
            "hyperparameters",
            "tuned_parameters",
        ],
    )

    filename_col = find_col(
        df,
        [
            "filename",
            "file",
            "model_file",
            "model_filename",
        ],
    )

    if model_col is None:
        raise ValueError(
            f"No model-name column found. Columns: {list(df.columns)}"
        )

    if require_params and params_col is None:
        raise ValueError(
            f"No Best Parameters column found. Columns: {list(df.columns)}"
        )

    result = pd.DataFrame({
        "dataset": (
            df[dataset_col].astype(str)
            if dataset_col
            else "multimodal_full"
        ),
        "model": df[model_col].astype(str),
    })

    if params_col:
        result["parameters"] = df[params_col].apply(parse_params)

    if filename_col:
        result["filename"] = df[filename_col].astype(str)

    return result


parameter_source_name = None
parameter_table = None

for preferred in [
    "candidate_models",
    "hyperparameter_tuning_summary",
]:
    if (
        preferred not in week5_tables
        or week5_tables[preferred].empty
    ):
        continue

    try:
        candidate = standardize_model_table(
            week5_tables[preferred],
            require_params=True,
        )

        if candidate["parameters"].map(bool).all():
            parameter_source_name = preferred
            parameter_table = candidate
            break

    except ValueError:
        continue


if parameter_table is None:
    raise ValueError(
        "Could not recover non-empty Week 5 hyperparameters from "
        "candidate_models.csv or hyperparameter_tuning_summary.csv. "
        "Validation will not continue with empty {} parameter dictionaries."
    )


tuned_models = parameter_table.copy()

# Join optional filenames separately.
if (
    "tuned_models" in week5_tables
    and not week5_tables["tuned_models"].empty
):
    inventory = standardize_model_table(
        week5_tables["tuned_models"]
    )

    if "filename" in inventory.columns:
        tuned_models = tuned_models.merge(
            inventory[
                ["dataset", "model", "filename"]
            ].drop_duplicates(),
            on=["dataset", "model"],
            how="left",
        )


tuned_models = tuned_models.drop_duplicates(
    subset=["dataset", "model"]
).reset_index(drop=True)

print(
    f"Using hyperparameters from: "
    f"{parameter_source_name}.csv"
)
print(
    "Empty parameter dictionaries:",
    int(
        (~tuned_models["parameters"].map(bool)).sum()
    ),
)

display(tuned_models)


## 4. Find the training and validation datasets

In [ ]:
# Optional manual overrides.
# Only fill these in if automatic discovery does not find the correct files.
MANUAL_DATA_FILES = {
    # Example:
    # "multimodal_full": {
    #     "train": DATA_DIR / "processed" / "multimodal_full_train.csv",
    #     "validation": DATA_DIR / "processed" / "multimodal_full_validation.csv",
    # }
}

all_csvs = []

for root in [DATA_DIR, OUTPUTS_DIR]:
    if root.exists():
        all_csvs.extend(root.rglob("*.csv"))

def clean_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def choose_file(dataset, kind):
    # Manual override wins.
    if dataset in MANUAL_DATA_FILES:
        value = MANUAL_DATA_FILES[dataset].get(kind)
        if value:
            p = Path(value)
            return p if p.exists() else None

    ds = clean_text(dataset)

    kind_words = {
        "train": ["train", "training"],
        "validation": ["validation", "valid", "val"],
    }[kind]

    candidates = []

    for p in all_csvs:
        name = clean_text(p.stem)

        # Ignore Week 5/Week 6 metric outputs.
        if any(
            bad in name
            for bad in [
                "candidate_models",
                "tuned_models",
                "hyperparameter_tuning",
                "validation_results",
                "confusion_matrix",
                "performance_summary",
                "metrics",
            ]
        ):
            continue

        if not any(word in name for word in kind_words):
            continue

        score = 0

        if ds and ds in name:
            score += 100

        ds_tokens = set(ds.split("_"))
        name_tokens = set(name.split("_"))
        score += len(ds_tokens & name_tokens)

        candidates.append((score, p))

    candidates.sort(key=lambda x: x[0], reverse=True)

    if candidates:
        return candidates[0][1]

    return None

data_records = []

for dataset in tuned_models["dataset"].unique():
    data_records.append({
        "dataset": dataset,
        "train_file": choose_file(dataset, "train"),
        "validation_file": choose_file(dataset, "validation"),
    })

data_inventory = pd.DataFrame(data_records)

display(data_inventory)

missing = data_inventory[
    data_inventory["train_file"].isna()
    | data_inventory["validation_file"].isna()
]

if not missing.empty:
    print("\nAutomatic discovery could not find every required train/validation file.")
    print("Available CSV files that contain 'train' or 'valid':")

    for p in all_csvs:
        n = p.name.lower()
        if "train" in n or "valid" in n:
            print(" -", p.relative_to(PROJECT_ROOT))

    print(
        "\nIf the correct files are listed above, add their paths to "
        "MANUAL_DATA_FILES and rerun this section."
    )

## 5. Recreate the Week 5 tuned pipeline exactly

The saved `Best Parameters` dictionary contains the tuned values, while fixed settings are recovered from the shared modeling configuration.

The validation pipeline is rebuilt in the same Week 5 order:

`preprocessing → variance_filter → feature_selection → classifier`

Specifically:

- `preprocessing` comes from `src.modeling.preprocessing.prepare_dataset`;
- `variance_filter` is `VarianceThreshold(threshold=0.0)`;
- `feature_selection` is `SelectKBest(score_func=f_classif, k=<saved Week 5 k>)`;
- `classifier` is recreated using the saved tuned classifier parameters plus the same fixed project settings.

The notebook intentionally raises an error if `feature_selection__k` is missing, because silently omitting that stage would no longer represent the Week 5 tuned model.


In [ ]:
FEATURE_SELECTION_KEYS = [
    "feature_selection__k",
    "select__k",
    "selector__k",
    "k",
    "n_selected_features",
]

VARIANCE_KEYS = [
    "variance_filter__threshold",
    "variance_threshold",
]


def strip_classifier_prefix(key):
    key = str(key)

    for prefix in [
        "classifier__",
        "model__",
        "estimator__",
        "clf__",
    ]:
        if key.startswith(prefix):
            return key[len(prefix):]

    return key


def get_selected_k(params):
    for key in FEATURE_SELECTION_KEYS:
        if key in params:
            value = params[key]

            if isinstance(value, str) and value.lower() == "all":
                return "all"

            return int(value)

    raise ValueError(
        "Week 5 feature_selection__k could not be recovered. "
        "Validation will not continue with a different feature-selection configuration."
    )


def get_variance_threshold(params):
    for key in VARIANCE_KEYS:
        if key in params:
            value = float(params[key])

            if value != 0.0:
                raise ValueError(
                    "Saved variance threshold does not match the Week 5 "
                    f"zero-variance filter: {value}"
                )

            return value

    # Week 5 used a fixed zero-variance filter rather than tuning this value.
    return 0.0


def classifier_params_only(params):
    cleaned = {}

    for key, value in params.items():
        key_text = str(key)

        if key in FEATURE_SELECTION_KEYS or key in VARIANCE_KEYS:
            continue

        if key_text.startswith(
            (
                "preprocessing__",
                "preprocess__",
                "variance_filter__",
                "feature_selection__",
                "select__",
                "selector__",
            )
        ):
            continue

        cleaned[strip_classifier_prefix(key)] = value

    return cleaned


def make_classifier(model_name, params):
    name = clean_text(model_name)
    params = classifier_params_only(params)

    # These fixed values mirror the shared project modeling configuration.
    class_weight = CLASS_WEIGHT if USE_CLASS_WEIGHT else None

    if "dummy" in name:
        return DummyClassifier(**params)

    if "logistic" in name:
        params.setdefault("solver", "lbfgs")
        params.setdefault("max_iter", LOGISTIC_MAX_ITER)
        params.setdefault("random_state", RANDOM_STATE)
        params.setdefault("class_weight", class_weight)
        return LogisticRegression(**params)

    if "random_forest" in name or "randomforest" in name:
        params.setdefault("random_state", RANDOM_STATE)
        params.setdefault("class_weight", class_weight)
        return RandomForestClassifier(**params)

    if "xgboost" in name or name in {"xgb", "xgbclassifier"}:
        params.setdefault("random_state", RANDOM_STATE)
        params.setdefault("objective", "multi:softprob")
        params.setdefault("eval_metric", "mlogloss")
        params.setdefault("n_jobs", -1)
        return XGBClassifier(**params)

    if "extra_trees" in name or "extratrees" in name:
        params.setdefault("random_state", RANDOM_STATE)
        params.setdefault("class_weight", class_weight)
        return ExtraTreesClassifier(**params)

    if "decision_tree" in name or name in {"tree", "dt"}:
        params.setdefault("random_state", RANDOM_STATE)
        params.setdefault("class_weight", class_weight)
        return DecisionTreeClassifier(**params)

    if "hist_gradient" in name or "histgradient" in name:
        params.setdefault("random_state", RANDOM_STATE)
        return HistGradientBoostingClassifier(**params)

    if "gradient_boost" in name or "gradientboost" in name:
        params.setdefault("random_state", RANDOM_STATE)
        return GradientBoostingClassifier(**params)

    if (
        name in {"svc", "svm", "support_vector_machine"}
        or "support_vector" in name
    ):
        params.setdefault("probability", True)
        return SVC(**params)

    if "knn" in name or "nearest_neighbor" in name:
        return KNeighborsClassifier(**params)

    raise ValueError(
        f"Model '{model_name}' is not mapped in make_classifier()."
    )


week5_configuration_audit = tuned_models[
    ["dataset", "model", "parameters"]
].copy()

week5_configuration_audit["variance_threshold"] = (
    week5_configuration_audit["parameters"].apply(get_variance_threshold)
)

week5_configuration_audit["feature_selection__k"] = (
    week5_configuration_audit["parameters"].apply(get_selected_k)
)

week5_configuration_audit["classifier_parameters"] = (
    week5_configuration_audit["parameters"].apply(classifier_params_only)
)

display(
    week5_configuration_audit[
        [
            "dataset",
            "model",
            "variance_threshold",
            "feature_selection__k",
            "classifier_parameters",
        ]
    ]
)


## 6. Fit on training data and evaluate on held-out validation

For every Week 5 candidate, the notebook reconstructs the same training pipeline and fits it only on the corresponding training data.

The validation data are never supplied to `.fit()`. They are used only after the full Week 5 pipeline has been fitted:

1. shared preprocessing;
2. zero-variance filtering;
3. tuned `SelectKBest`;
4. tuned classifier;
5. `predict()` and, where available, `predict_proba()` on held-out validation.

An exactness audit is recorded for every successful candidate so the pipeline steps, variance threshold, selected `k`, classifier class, and classifier parameters can be inspected directly.


In [ ]:
results = []
prediction_store = {}
prediction_rows = []
probability_rows = []
week5_exactness_rows = []


for _, model_row in tuned_models.iterrows():
    dataset = model_row["dataset"]
    model_name = model_row["model"]
    params = model_row["parameters"]

    data_row = data_inventory[
        data_inventory["dataset"] == dataset
    ]

    if data_row.empty:
        print(
            f"✗ {dataset} | {model_name}: "
            "no dataset mapping"
        )
        continue

    train_file = data_row.iloc[0]["train_file"]
    validation_file = (
        data_row.iloc[0]["validation_file"]
    )

    if (
        pd.isna(train_file)
        or pd.isna(validation_file)
    ):
        print(
            f"✗ {dataset} | {model_name}: "
            "train/validation file missing"
        )
        continue

    try:
        train_df = pd.read_csv(
            train_file,
            dtype={ID_COLUMN: str},
        )

        validation_df = pd.read_csv(
            validation_file,
            dtype={ID_COLUMN: str},
        )

        if TARGET not in train_df.columns:
            raise ValueError(
                f"{TARGET!r} missing "
                "from training data"
            )

        if TARGET not in validation_df.columns:
            raise ValueError(
                f"{TARGET!r} missing "
                "from validation data"
            )

        # Shared project feature selection/exclusion
        # and preprocessing configuration.
        X_train, y_train, preprocessing = (
            prepare_dataset(train_df)
        )

        X_val, y_val, _ = (
            prepare_dataset(validation_df)
        )

        # Align validation columns to the exact
        # training feature schema.
        missing_in_validation = [
            c
            for c in X_train.columns
            if c not in X_val.columns
        ]

        if missing_in_validation:
            raise ValueError(
                "Validation data are missing "
                f"{len(missing_in_validation)} "
                "training feature(s): "
                f"{missing_in_validation[:10]}"
            )

        X_val = X_val[
            X_train.columns
        ].copy()

        classifier = make_classifier(
            model_name,
            params,
        )

        selected_k = get_selected_k(params)
        variance_threshold = get_variance_threshold(params)

        pipeline_steps = [
            (
                "preprocessing",
                preprocessing,
            ),
            (
                "variance_filter",
                VarianceThreshold(
                    threshold=variance_threshold,
                ),
            ),
        ]

        if selected_k is not None:
            pipeline_steps.append(
                (
                    "feature_selection",
                    SelectKBest(
                        score_func=f_classif,
                        k=selected_k,
                    ),
                )
            )

        pipeline_steps.append(
            (
                "classifier",
                classifier,
            )
        )

        pipeline = Pipeline(
            pipeline_steps
        )

        expected_steps = [
            "preprocessing",
            "variance_filter",
            "feature_selection",
            "classifier",
        ]

        actual_steps = [name for name, _ in pipeline.steps]

        if actual_steps != expected_steps:
            raise AssertionError(
                f"Week 5 pipeline mismatch for {dataset} | {model_name}. "
                f"Expected {expected_steps}, got {actual_steps}."
            )

        week5_exactness_rows.append({
            "dataset": dataset,
            "model": model_name,
            "pipeline_steps": " -> ".join(actual_steps),
            "variance_threshold": variance_threshold,
            "feature_selection__k": selected_k,
            "classifier_class": type(classifier).__name__,
            "classifier_params": classifier.get_params(deep=False),
            "parameter_source": parameter_source_name,
            "status": "PASS",
        })

        # Training information only.
        pipeline.fit(
            X_train,
            y_train,
        )

        y_pred = pipeline.predict(X_val)

        metrics = calculate_metrics(
            y_val,
            y_pred,
        )

        key = (
            f"{clean_text(dataset)}"
            f"__{clean_text(model_name)}"
        )

        prediction_store[key] = (
            y_val.copy(),
            y_pred.copy(),
        )

        results.append({
            "result_key": key,
            "dataset": dataset,
            "model": model_name,
            "n_validation": len(y_val),
            "variance_threshold": variance_threshold,
            "feature_selection__k": selected_k,
            **metrics,
        })

        if ID_COLUMN in validation_df.columns:
            participant_ids = (
                validation_df[
                    ID_COLUMN
                ].astype(str).values
            )
        else:
            participant_ids = (
                np.arange(
                    len(validation_df)
                ).astype(str)
            )

        for (
            participant_id,
            true_label,
            predicted_label,
        ) in zip(
            participant_ids,
            y_val,
            y_pred,
        ):
            prediction_rows.append({
                ID_COLUMN: participant_id,
                "dataset": dataset,
                "model": model_name,
                "true_label": true_label,
                "predicted_label": predicted_label,
            })

        # Save probabilities for calibration,
        # Brier score, ROC/PR, SHAP, etc.
        if hasattr(
            pipeline,
            "predict_proba",
        ):
            probabilities = (
                pipeline.predict_proba(
                    X_val
                )
            )

            classes = pipeline.classes_

            for row_index, (
                participant_id,
                true_label,
            ) in enumerate(
                zip(
                    participant_ids,
                    y_val,
                )
            ):
                probability_row = {
                    ID_COLUMN: participant_id,
                    "dataset": dataset,
                    "model": model_name,
                    "true_label": true_label,
                }

                for (
                    class_index,
                    class_label,
                ) in enumerate(classes):
                    probability_row[
                        f"prob_class_{class_label}"
                    ] = probabilities[
                        row_index,
                        class_index,
                    ]

                probability_rows.append(
                    probability_row
                )

        else:
            print(
                f"⚠ {dataset} | {model_name}: "
                "predict_proba() unavailable"
            )

        print(
            f"✓ {dataset} | {model_name} | "
            f"variance_threshold={variance_threshold} | "
            f"k={selected_k} | "
            f"Macro F1="
            f"{metrics['macro_f1']:.4f}"
        )

    except Exception as exc:
        print(
            f"✗ {dataset} | {model_name}: "
            f"{type(exc).__name__}: {exc}"
        )


validation_results = pd.DataFrame(
    results
)

if validation_results.empty:
    raise RuntimeError(
        "No validation model completed "
        "successfully."
    )


validation_results = (
    validation_results
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

validation_results.insert(
    0,
    "rank",
    np.arange(
        1,
        len(validation_results) + 1,
    ),
)


validation_predictions = pd.DataFrame(
    prediction_rows
)

validation_probabilities = pd.DataFrame(
    probability_rows
)


validation_predictions.to_csv(
    METRICS_DIR
    / "validation_predictions.csv",
    index=False,
)

validation_probabilities.to_csv(
    METRICS_DIR
    / "validation_probabilities.csv",
    index=False,
)


print(
    "Saved class-prediction rows:",
    len(validation_predictions),
)

print(
    "Saved probability rows:",
    len(validation_probabilities),
)

display(validation_results)

week5_pipeline_exactness_audit = pd.DataFrame(
    week5_exactness_rows
)

week5_pipeline_exactness_audit.to_csv(
    TABLES_DIR / "week5_pipeline_exactness_audit.csv",
    index=False,
)

if len(week5_pipeline_exactness_audit) != len(validation_results):
    raise AssertionError(
        "Week 5 exactness audit row count does not match completed validation models."
    )

assert (
    week5_pipeline_exactness_audit["status"]
    .eq("PASS")
    .all()
)

print(
    "Week 5 pipeline exactness audit: PASS for "
    f"{len(week5_pipeline_exactness_audit)} completed candidate(s)."
)

display(week5_pipeline_exactness_audit)



## 7. Save `validation_results.csv` and comparison table

In [ ]:
validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

comparison_cols = [
    "rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison = validation_results[
    comparison_cols
].copy()

validation_comparison.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

display(validation_comparison.round(4))

## 8. Validation confusion matrices

In [ ]:
# Generate and save validation confusion matrices as tables

for _, row in validation_results.iterrows():
    key = row["result_key"]
    y_true, y_pred = prediction_store[key]

    labels = sorted(
        pd.Series(y_true).dropna().unique().tolist()
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {x}" for x in labels],
        columns=[f"Predicted {x}" for x in labels]
    )

    # Save confusion matrix
    csv_path = (
        METRICS_DIR /
        f"{key}_validation_confusion_matrix.csv"
    )

    cm_df.to_csv(csv_path)

    # Show it directly in the notebook
    print(f"\n{row['dataset']} — {row['model']}")
    display(cm_df)

    print(f"Saved: {csv_path.name}")

## 9. Validation comparison figure

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

plot_df = validation_results[
    ["dataset", "model", "macro_f1", "balanced_accuracy", "accuracy"]
].copy()

plot_df["candidate"] = (
    plot_df["dataset"].astype(str)
    + " | "
    + plot_df["model"].astype(str)
)

plot_df = plot_df.sort_values(
    ["macro_f1", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

figure_path = Path(FIGURES_DIR) / "validation_model_comparison.png"

img = Image.new("RGB", (1200, 850), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default()

draw.text((40, 30), "Validation Model Comparison", fill="black", font=font)
draw.text(
    (40, 55),
    "Models ranked by validation Macro F1. Higher scores are better.",
    fill="black",
    font=font
)

bar_x = 430
bar_width = 650

for i, row in plot_df.iterrows():
    y = 110 + i * 110

    label = f"#{i+1} {row['candidate']}"
    if i == 0:
        label += " - BEST"

    draw.text((40, y), label, fill="black", font=font)

    draw.rectangle(
        [bar_x, y + 25, bar_x + bar_width, y + 50],
        fill="lightgray"
    )

    score_width = int(float(row["macro_f1"]) * bar_width)

    draw.rectangle(
        [bar_x, y + 25, bar_x + score_width, y + 50],
        fill="steelblue"
    )

    metrics = (
        f"Macro F1: {row['macro_f1']:.4f} | "
        f"Balanced Acc: {row['balanced_accuracy']:.4f} | "
        f"Accuracy: {row['accuracy']:.4f}"
    )

    draw.text((40, y + 60), metrics, fill="black", font=font)

img.save(str(figure_path), format="PNG")

print("Saved:", figure_path)
print("Exists:", figure_path.exists())

## 10. Select the best candidate model(s)

In [ ]:
best_by_dataset = (
    validation_results
    .sort_values(
        [
            "dataset",
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[True, False, False],
    )
    .groupby(
        "dataset",
        as_index=False,
    )
    .first()
)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

display(
    best_by_dataset[
        [
            "dataset",
            "model",
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
        ]
    ].round(4)
)

best = validation_results.iloc[0]

print(
    f"Best overall candidate: "
    f"{best['model']} ({best['dataset']})"
)
print(
    f"Macro F1: {best['macro_f1']:.4f}"
)
print(
    f"Balanced accuracy: "
    f"{best['balanced_accuracy']:.4f}"
)

## 11. Validation metrics summary and justification

In [ ]:
display(
    validation_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

print("\nJUSTIFICATION")
print(
    f"{best['model']} on {best['dataset']} "
    f"is the strongest candidate to move forward "
    f"because it achieved the highest validation "
    f"Macro F1 ({best['macro_f1']:.4f}). "
    f"Its balanced accuracy was "
    f"{best['balanced_accuracy']:.4f}. "
    f"Macro F1 is used as the primary selection "
    f"metric because it gives equal importance to "
    f"performance across classes, while balanced "
    f"accuracy is used as the secondary comparison."
)

## 12. Deliverables Check

In [ ]:
from pathlib import Path
import pandas as pd

# Required validation deliverables
deliverables = pd.DataFrame([
    {
        "deliverable": "Validation results",
        "path": METRICS_DIR / "validation_results.csv",
    },
    {
        "deliverable": "Validation class predictions",
        "path": METRICS_DIR / "validation_predictions.csv",
    },
    {
        "deliverable": "Validation probability predictions",
        "path": METRICS_DIR / "validation_probabilities.csv",
    },
    {
        "deliverable": "Week 5 pipeline exactness audit",
        "path": TABLES_DIR / "week5_pipeline_exactness_audit.csv",
    },
    {
        "deliverable": "Validation comparison table",
        "path": TABLES_DIR / "validation_comparison.csv",
    },
    {
        "deliverable": "Best candidate table",
        "path": TABLES_DIR / "validation_best_models.csv",
    },
    {
        "deliverable": "Validation comparison figure",
        "path": FIGURES_DIR / "validation_model_comparison.png",
    },
])

deliverables["status"] = deliverables["path"].apply(
    lambda p: "READY" if Path(p).exists() else "MISSING"
)

display(deliverables)

confusion_matrices = list(
    METRICS_DIR.glob("*_validation_confusion_matrix.csv")
)

print(
    f"Validation confusion matrices saved: "
    f"{len(confusion_matrices)}"
)

print(
    "Probability rows saved:",
    len(validation_probabilities),
)

if (deliverables["status"] == "READY").all():
    print()
    print("✓ All required validation outputs are ready.")
else:
    print()
    print("⚠ Some required validation outputs are missing.")